In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

# ==================== 1. CARGAR DATOS ====================
df_raw = pd.read_csv(
    '/home/juanchx/Documents/Trabajo/SYSTEM_RECOMENDATION_FNN/Data/Historico_08122025.csv',
    sep=';', encoding='latin-1'
)

df_raw['DIM_PERIODO'] = pd.to_datetime(df_raw['DIM_PERIODO'], format='%d-%b-%y')

# Agrupar por FAMILIA y SUBCATEGORIA
df_agg = df_raw.groupby(['CODIGO_FAMILIA', 'COD_SUBCATEGORIA', 'DIM_PERIODO']).agg({
    'CANTIDAD_SUELTA': 'sum'
}).reset_index()

df_agg = df_agg.sort_values(['CODIGO_FAMILIA', 'COD_SUBCATEGORIA', 'DIM_PERIODO'])

print(f"Series: {df_agg.groupby(['CODIGO_FAMILIA', 'COD_SUBCATEGORIA']).ngroups}")

# ==================== 2. CREAR SECUENCIAS (SIN SLIDING WINDOWS) ====================
# Ventana de 10 "timesteps" => aquí timestep = compra (realmente intervalos entre compras)
MAX_STEPS = 10                 # máximo de deltas (Δt) en la secuencia
MIN_COMPRAS = 4                # mínimo compras totales para tener al menos 3 en input + 1 target
N_FEATURES = 3                 # [log1p(Δt), log1p(qty), qty_z_serie]

X_list = []
y_list_log = []
y_list_days = []
metadata_list = []

for (familia, subcat), grupo in df_agg.groupby(['CODIGO_FAMILIA', 'COD_SUBCATEGORIA']):
    grupo = grupo.sort_values('DIM_PERIODO').copy()

    # Necesitas al menos 4 compras: 3 en input + 1 como target
    if len(grupo) < MIN_COMPRAS:
        continue


    if len(grupo) > (MAX_STEPS):
        grupo_input = grupo.iloc[-(MAX_STEPS):-1].copy()  # input (compras)
        siguiente_compra_fecha = grupo.iloc[-1]['DIM_PERIODO']  # target fecha
    else:
        grupo_input = grupo.iloc[:-1].copy()
        siguiente_compra_fecha = grupo.iloc[-1]['DIM_PERIODO']

    # group_input debe tener al menos 3 compras para dar al menos 2 deltas
    if len(grupo_input) < 3:
        continue

    # ===== Features basadas en INTERVALOS (Δt) =====
    fechas_in = grupo_input['DIM_PERIODO'].values
    qty_in = grupo_input['CANTIDAD_SUELTA'].values

    # Δt entre compras consecutivas dentro del INPUT
    deltas = np.diff(fechas_in).astype('timedelta64[D]').astype(int)  # len = n_input - 1

    # Cantidades alineadas con cada delta:
    # delta[i] es de compra i -> compra i+1, usamos qty de la compra i+1 (la "llegada")
    qty_aligned = qty_in[1:]  # misma longitud que deltas

    # Transformaciones robustas (sin scalers)
    x_time = np.log1p(np.clip(deltas, 0, None)).astype(np.float32)
    x_qty = np.log1p(np.clip(qty_aligned, 0, None)).astype(np.float32)

    # z-score dentro de la serie (opcional pero lo dejo fijo como feature)
    q_mean = float(np.mean(qty_aligned))
    q_std = float(np.std(qty_aligned))
    if q_std < 1e-8:
        x_qz = np.zeros_like(x_qty, dtype=np.float32)
    else:
        x_qz = ((qty_aligned - q_mean) / q_std).astype(np.float32)

    # Secuencia (timesteps = deltas)
    X_seq = np.column_stack([x_time, x_qty, x_qz]).astype(np.float32)

    # Padding al INICIO con ceros (log1p(0)=0 => padding natural)
    seq_len = len(X_seq)
    if seq_len < MAX_STEPS:
        pad = np.zeros((MAX_STEPS - seq_len, N_FEATURES), dtype=np.float32)
        X_seq = np.vstack([pad, X_seq])
    elif seq_len > MAX_STEPS:
        # Por seguridad: recorta por la derecha (más reciente)
        X_seq = X_seq[-MAX_STEPS:, :]

    # ===== Target: días hasta la siguiente compra (desde la última compra del input) =====
    ultima_fecha_input = grupo_input['DIM_PERIODO'].iloc[-1]
    target_days = int((siguiente_compra_fecha - ultima_fecha_input).days)
    if target_days < 0:
        # Esto no debería pasar si está ordenado, pero lo protegemos
        continue

    y_log = np.log1p(target_days).astype(np.float32)

    X_list.append(X_seq)
    y_list_log.append(y_log)
    y_list_days.append(target_days)

    metadata_list.append({
        'familia': familia,
        'subcategoria': subcat,
        'n_compras_input': len(grupo_input),
        'ultima_fecha': ultima_fecha_input,
        'siguiente_fecha': siguiente_compra_fecha,
        'target_days': target_days
    })

X = np.array(X_list, dtype=np.float32)
y_log = np.array(y_list_log, dtype=np.float32)
y_days = np.array(y_list_days, dtype=np.int32)

print(f"\n{'='*60}")
print("DATASET FINAL")
print(f"{'='*60}")
print(f"X shape: {X.shape}  (samples, {MAX_STEPS}, {N_FEATURES})")
print(f"y_log shape: {y_log.shape}")
print(f"Total series: {len(X)}")
print(f"Target días - min: {y_days.min()}, max: {y_days.max()}, mean: {y_days.mean():.1f}")

# ==================== 3. SPLIT 80/20 (SIN LEAKAGE) ====================
X_train, X_test, y_train, y_test, y_train_days, y_test_days, meta_train, meta_test = train_test_split(
    X, y_log, y_days, metadata_list, test_size=0.2, random_state=42
)

print(f"\nTrain: {len(X_train)}")
print(f"Test: {len(X_test)}")

# ==================== 4. ARQUITECTURA RNN ====================
model = keras.Sequential([
    layers.Masking(mask_value=0.0, input_shape=(MAX_STEPS, N_FEATURES)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)  # regresión en espacio log1p(días)
])

model.compile(
    optimizer='adam',
    loss=keras.losses.Huber(),   # más robusto que MSE para colas largas
    metrics=['mae']
)

model.summary()

# ==================== 5. ENTRENAR ====================
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

# ==================== 6. EVALUAR (en escala original de días) ====================
y_pred_log = model.predict(X_test, verbose=0).flatten()

y_pred_days = np.expm1(y_pred_log)   # inverse log1p
y_true_days = np.expm1(y_test)       # inverse log1p

mae = np.mean(np.abs(y_pred_days - y_true_days))
rmse = np.sqrt(np.mean((y_pred_days - y_true_days) ** 2))

print(f"\n{'='*60}")
print("RESULTADOS")
print(f"{'='*60}")
print(f"MAE: {mae:.2f} días")
print(f"RMSE: {rmse:.2f} días")

for i in range(5):
    print(f"\nEjemplo {i+1}:")
    print(f"  Real: {y_true_days[i]:.1f} días")
    print(f"  Predicho: {y_pred_days[i]:.1f} días")
    print(f"  Error: {abs(y_pred_days[i] - y_true_days[i]):.1f} días")

# ==================== 7. GRÁFICAS ====================
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Loss
axes[0, 0].plot(history.history['loss'], label='Train')
axes[0, 0].plot(history.history['val_loss'], label='Val')
axes[0, 0].set_title('Loss (Huber)')
axes[0, 0].legend()
axes[0, 0].grid(True)

# MAE (en espacio log, porque es la métrica que calcula Keras)
axes[0, 1].plot(history.history['mae'], label='Train')
axes[0, 1].plot(history.history['val_mae'], label='Val')
axes[0, 1].set_title('MAE (log1p space)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Real vs Predicho (días)
axes[1, 0].scatter(y_true_days, y_pred_days, alpha=0.5)
mn = min(y_true_days.min(), y_pred_days.min())
mx = max(y_true_days.max(), y_pred_days.max())
axes[1, 0].plot([mn, mx], [mn, mx], 'r--')
axes[1, 0].set_xlabel('Real (días)')
axes[1, 0].set_ylabel('Predicho (días)')
axes[1, 0].set_title('Real vs Predicho (días)')
axes[1, 0].grid(True)

# Distribución de errores (días)
errors = y_pred_days - y_true_days
axes[1, 1].hist(errors, bins=50, edgecolor='black')
axes[1, 1].set_xlabel('Error (días)')
axes[1, 1].set_title(f'Distribución de Errores (días)\nMAE={mae:.1f}')
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('/home/juanchx/Documents/Trabajo/SYSTEM_RECOMENDATION_FNN/src/Recurencia/rnn_regression_results.png', dpi=300)
print("\nGráficas guardadas")

model.save('/home/juanchx/Documents/Trabajo/SYSTEM_RECOMENDATION_FNN/src/Recurencia/rnn_regression_model.h5')
print("Modelo guardado")


2026-01-13 17:39:52.058737: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-13 17:39:52.261004: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-13 17:39:52.262084: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-13 17:39:53.907077: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
/tmp/ipykernel_8491/1798702383.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(


Series: 282683


KeyboardInterrupt: 